# Toxicity Classification with PyTorch

In [16]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import pandas as pd

## Load Jigsaw Toxicity Dataset
Load the [Jigsaw Toxic Comment Classification](https://huggingface.co/datasets/Arsive/toxicity_classification_jigsaw) dataset from Hugging Face.

In [17]:
from datasets import load_dataset

ds = load_dataset("Arsive/toxicity_classification_jigsaw")

train_df = ds["train"].to_pandas()
train_df = train_df[train_df["toxic"]!=-1]

test_df = ds["test"].to_pandas()
test_df = test_df[test_df["toxic"]!=-1]

print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

Training set size: 25960
Test set size: 63978


In [6]:
train_df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,17dd17eb535029d6,"""\n\nDear Jesus. Really, has Vander Plaats rea...",1,0,1,0,1,1
1,9fc74022287f6325,"You NDP attack queers, sorry, Cabal of Sanctim...",0,0,0,0,0,0
2,1f64687358ed17f0,Tree thinking eliminates major swats of what e...,0,0,0,0,0,0
3,ad339ac2d862a2d1,"""\n\n Eternity clause \n\nYou are right about ...",0,0,0,0,0,0
4,15c86e4139f26111,Unspecified source for Image:Nana.JPG\n\nThank...,0,0,0,0,0,0


## Model Setup
Define a simple PyTorch classifier using a pre-trained transformer.

In [18]:
class ToxicityClassifier(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased"):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, 1)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]  # [CLS] token
        return self.classifier(pooled)

# Initialize model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = ToxicityClassifier()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2192.50it/s, Materializing param=transformer.layer.5.sa_layer_norm.weight]   
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
class SimpleToxicityClassifier(nn.Module):
    def __init__(self, vocab_size=10000, embedding_dim=128, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.hidden = nn.Linear(embedding_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)
        self.relu = nn.ReLU()
        
    def forward(self, input_ids):
        # input_ids: [batch_size, seq_len]
        embedded = self.embedding(input_ids)  # [batch_size, seq_len, embedding_dim]
        pooled = embedded.mean(dim=1)  # Simple average pooling
        hidden = self.relu(self.hidden(pooled))
        return self.classifier(hidden)

In [20]:
class MiniTransformerEncoderClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        nhead=4,
        num_layers=1,
        hidden_dim=64,
        max_len=100,
        dropout=0.1,
    ):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.position_embedding = nn.Embedding(max_len, embedding_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=nhead,
            dim_feedforward=embedding_dim * 2,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.hidden = nn.Linear(embedding_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, seq_len)

        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x = self.dropout(x)
        x = self.encoder(x, src_key_padding_mask=(input_ids == 0))

        pooled = x.mean(dim=1)
        hidden = self.relu(self.hidden(pooled))
        return self.classifier(hidden)

## Training

In [21]:
# Training setup
from torch.utils.data import TensorDataset, DataLoader

# Prepare a small training subset for demonstration
train_subset = train_df.sample(n=1000, random_state=42)
train_texts = train_subset["comment_text"].tolist()
train_labels = torch.tensor(train_subset["toxic"].values, dtype=torch.float32)

In [22]:
test_subset = test_df.sample(n=1000,random_state=42)
test_texts = test_subset["comment_text"].tolist()
test_labels = torch.tensor(test_subset["toxic"].values, dtype=torch.float32)

In [23]:
# Train SimpleToxicityClassifier
from torch.nn.utils.rnn import pad_sequence

# Simple tokenizer: build vocab from top words
from collections import Counter
all_words = ' '.join(train_texts).lower().split()
vocab = {word: idx+1 for idx, (word, _) in enumerate(Counter(all_words).most_common(9999))}

def simple_tokenize(texts, vocab, max_len=100):
    token_ids = []
    for text in texts:
        tokens = [vocab.get(word, 0) for word in text.lower().split()[:max_len]]
        token_ids.append(torch.tensor(tokens))
    return pad_sequence(token_ids, batch_first=True, padding_value=0)

# Train simple model
simple_model = SimpleToxicityClassifier(vocab_size=len(vocab)+1)
optimizer = torch.optim.Adam(simple_model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

simple_model.train()
train_ids = simple_tokenize(train_texts, vocab)
for epoch in range(3):
    optimizer.zero_grad()
    logits = simple_model(train_ids).squeeze()
    loss = criterion(logits, train_labels)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.6912
Epoch 2, Loss: 0.6897
Epoch 3, Loss: 0.6881


In [24]:
# Train ToxicityClassifier (DistilBERT)
model = ToxicityClassifier()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.BCEWithLogitsLoss()

model.train()
batch_size = 8
num_epochs = 2

for epoch in range(num_epochs):
    total_loss = 0
    for i in range(0, len(train_texts), batch_size):
        batch_texts = train_texts[i:i+batch_size]
        batch_labels = train_labels[i:i+batch_size]
        
        encoded = tokenizer(batch_texts, padding=True, truncation=True, 
                          max_length=128, return_tensors="pt")
        
        optimizer.zero_grad()
        logits = model(encoded["input_ids"], encoded["attention_mask"]).squeeze()
        loss = criterion(logits, batch_labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / (len(train_texts) // batch_size)
    print(f"Epoch {epoch+1}/{num_epochs}, Avg Loss: {avg_loss:.4f}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2293.90it/s, Materializing param=transformer.layer.5.sa_layer_norm.weight]   
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/2, Avg Loss: 0.4254
Epoch 2/2, Avg Loss: 0.1727


In [25]:
# Train MiniTransformerEncoderClassifier
encoder_model = MiniTransformerEncoderClassifier(vocab_size=len(vocab) + 1)
encoder_optimizer = torch.optim.Adam(encoder_model.parameters(), lr=0.001)
encoder_criterion = nn.BCEWithLogitsLoss()

encoder_model.train()
for epoch in range(3):
    encoder_optimizer.zero_grad()
    logits = encoder_model(train_ids).squeeze()
    loss = encoder_criterion(logits, train_labels)
    loss.backward()
    encoder_optimizer.step()
    print(f"MiniTransformer Epoch {epoch+1}, Loss: {loss.item():.4f}")

MiniTransformer Epoch 1, Loss: 0.6996
MiniTransformer Epoch 2, Loss: 0.6886
MiniTransformer Epoch 3, Loss: 0.6821


## Evaluation Setup
Prepare evaluation metrics and inference pipeline.

In [26]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_distilbert_model(model, test_texts, test_labels, tokenizer, batch_size=32):
    model.eval()
    all_preds = []

    with torch.no_grad():
        for i in range(0, len(test_texts), batch_size):
            batch_texts = test_texts[i:i+batch_size]
            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt",
            )

            logits = model(encoded["input_ids"], encoded["attention_mask"]).squeeze()
            preds = (torch.sigmoid(logits) > 0.5).long().cpu().numpy()
            all_preds.extend(preds)

    y_true = test_labels.cpu().numpy() if torch.is_tensor(test_labels) else test_labels
    print(classification_report(y_true, all_preds))
    return all_preds


def evaluate_token_id_model(model, test_texts, test_labels, vocab, max_len=100):
    model.eval()
    test_ids = simple_tokenize(test_texts, vocab, max_len=max_len)

    with torch.no_grad():
        logits = model(test_ids).squeeze()
        preds = (torch.sigmoid(logits) > 0.5).long().cpu().numpy()

    y_true = test_labels.cpu().numpy() if torch.is_tensor(test_labels) else test_labels
    print(classification_report(y_true, preds))
    return preds

In [27]:
print("SimpleToxicityClassifier evaluation")
simple_preds = evaluate_token_id_model(simple_model, test_texts, test_labels, vocab)

print("DistilBERT ToxicityClassifier evaluation")
distilbert_preds = evaluate_distilbert_model(model, test_texts, test_labels, tokenizer)

print("MiniTransformerEncoderClassifier evaluation")
encoder_preds = evaluate_token_id_model(encoder_model, test_texts, test_labels, vocab)

SimpleToxicityClassifier evaluation
              precision    recall  f1-score   support

         0.0       0.90      1.00      0.95       897
         1.0       0.00      0.00      0.00       103

    accuracy                           0.90      1000
   macro avg       0.45      0.50      0.47      1000
weighted avg       0.80      0.90      0.85      1000

DistilBERT ToxicityClassifier evaluation


/Users/dskar/Documents/GitHub/ai-learning-tower/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/dskar/Documents/GitHub/ai-learning-tower/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/dskar/Documents/GitHub/ai-learning-tower/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control 

              precision    recall  f1-score   support

         0.0       1.00      0.71      0.83       897
         1.0       0.28      0.98      0.44       103

    accuracy                           0.74      1000
   macro avg       0.64      0.85      0.63      1000
weighted avg       0.92      0.74      0.79      1000

MiniTransformerEncoderClassifier evaluation
              precision    recall  f1-score   support

         0.0       0.91      0.91      0.91       897
         1.0       0.19      0.17      0.18       103

    accuracy                           0.84      1000
   macro avg       0.55      0.54      0.55      1000
weighted avg       0.83      0.84      0.84      1000

